# 🎓 cutrug – profi lépések videóból (3D mozgáskövetés)

Ez a jegyzetfüzet egy táncvideóból **teljes 3D mozgást** nyer ki (a táncos oldalra lépését, forgását, ugrását is) a nyílt forrású **GVHMR** modellel,
majd **ütemre, nyolcasokra** bontja – ugyanabban a formátumban, mint a játék profi (AIST++) lépései. A kimeneti `…_cutrug.json` fájlt a
**Tanuló stúdió** (`https://zorits1.github.io/zk-dance-party/?studio`) tölti be: ott a 3D figura eltáncolja, elnevezed, stílusba teszed, mented.

**Előtte (egyszer):**
1. Fent: **Futtatókörnyezet → Futtatókörnyezet típusának módosítása → T4 GPU** (ingyenes).
2. Ingyenes regisztráció a 3D testmodellhez: **https://smpl-x.is.tue.mpg.de** (és ajánlott: https://smpl.is.tue.mpg.de). A 2. cella kéri a belépést.

**Tipp a jó eredményhez:** egy táncos, tetőtől talpig, álló kamera, szemből; legalább 30–60 mp tánc, ahol a lépések ismétlődnek.
Csak saját vagy engedélyezett videót használj (a YouTube szabályai a letöltést nem engedik).

Futtatás: **Futtatókörnyezet → Mindegyik futtatása** (vagy cellánként ▶).

In [ ]:
#@title 1. Telepítés (kb. 3–6 perc)
!pip -q install "gvhmr[preproc]" librosa yt-dlp
!wget -q -O pro_moves.py https://raw.githubusercontent.com/zorits1/zk-dance-party/main/tools/pro_moves.py
!wget -q -O mocap_to_moves.py https://raw.githubusercontent.com/zorits1/zk-dance-party/main/tools/mocap_to_moves.py
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINCS – állítsd be: Futtatókörnyezet → T4 GPU')

In [ ]:
#@title 2. Belépés a 3D testmodellhez (SMPL-X – egyszer kell; a felhasználónevet és jelszót a cella alatt kéri)
!gvhmr auth smpl

In [ ]:
#@title 3. Videó és beállítások
VIDEO_URL = ""  #@param {type:"string"}
CIM = "Saját tánc"  #@param {type:"string"}
STILUS = "House"  #@param ["House","Deep House","Techno","EDM","Drum & Bass","Hip-hop / Rap","Trap","R&B","Pop","Rock","Metal","Latin / Reggaeton","Disco / Funk","Retro","Magyar","Gyerekdal"]
KEZDET_MP = 0  #@param {type:"number"}
HOSSZ_MP = 90  #@param {type:"number"}
ALLO_KAMERA = True  #@param {type:"boolean"}
import os, re, subprocess, json
os.makedirs('vid', exist_ok=True)
if VIDEO_URL.strip():
    # csak saját / engedélyezett videóval!
    subprocess.run(['yt-dlp', '-f', 'mp4[height<=720]/best[height<=720]/best', '-o', 'vid/src.%(ext)s', VIDEO_URL.strip()], check=True)
    src = [os.path.join('vid', f) for f in os.listdir('vid') if f.startswith('src.')][0]
else:
    from google.colab import files
    up = files.upload(); src = list(up.keys())[0]
YT = (re.search(r'(?:v=|youtu\.be/|shorts/)([\w-]{11})', VIDEO_URL) or [None, ''])[1]
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-ss', str(KEZDET_MP), '-i', src, '-t', str(HOSSZ_MP), '-vf', 'scale=-2:720', '-r', '30', '-c:v', 'libx264', '-c:a', 'aac', 'vid/clip.mp4'], check=True)
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', 'vid/clip.mp4', '-vn', '-ac', '1', '-ar', '22050', 'vid/clip.wav'], check=True)
print('kész:', src, '→ vid/clip.mp4 (30 kép/mp)')

In [ ]:
#@title 4. Ütem a zenéből
import librosa, numpy as np
y, sr = librosa.load('vid/clip.wav', sr=22050)
tempo, beats = librosa.beat.beat_track(y=y, sr=sr, units='time')
BPM = float(np.atleast_1d(tempo)[0])
while BPM < 80: BPM *= 2
while BPM > 160: BPM /= 2
T0 = float(beats[0]) if len(beats) else None
print(f'tempó: {BPM:.1f} BPM, első ütem: {T0}')

In [ ]:
#@title 5. 3D mozgáskövetés (GVHMR) – videóhossztól függően 1–5 perc
import gvhmr, numpy as np
pipe = gvhmr.pipeline('human-motion-recovery', device='cuda')
res = pipe('vid/clip.mp4', static_camera=ALLO_KAMERA)
J = np.asarray(res.joints_world)
print('ízületek:', J.shape)
np.savez('vid/joints.npz', joints=J)

In [ ]:
#@title 6. Nyolcasok és letöltés
from mocap_to_moves import smpl_to_coco, resample, smooth, extract
K = smooth(resample(smpl_to_coco(J), 30.0))
out = extract(K, BPM, STILUS, CIM, YT, t0=T0)
fn = re.sub(r'[^\w-]+', '_', CIM).strip('_')[:40] + '_cutrug.json'
json.dump(out, open(fn, 'w'), separators=(',', ':'))
print(f"{len(out['moves'])} lépés ({out['candidates']} jelöltből), {out['bpm']} BPM")
for m in out['moves']: print(f"  {m['t']:6.1f} mp  {m['n']}  (minőség {m['sc']}, oldalra {m['trav']})")
from google.colab import files; files.download(fn)
print('Töltsd be a Tanuló stúdióban: https://zorits1.github.io/zk-dance-party/?studio')